In [33]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain.agents import create_agent
from langchain_tavily import TavilySearch
from langchain_core.messages import HumanMessage
load_dotenv()

llm = ChatOpenAI(model='gpt-4o-mini')
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
tavily_tool = TavilySearch(max=5)

txt_file_path = r'C:\Users\visha\agentic-AI\notes_using_claude\agents-from-scratch\docs\crow_and_fox.txt'
with open(txt_file_path, encoding='utf-8', mode='r') as txt_file:
    txt_contents = txt_file.read()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
documents = splitter.create_documents([txt_contents])

vector_store = Chroma(
    collection_name='fox_crow_story',
    embedding_function=embeddings
)

vector_store.add_documents(
    documents=documents
)

retriever = vector_store.as_retriever(
    search_type = 'mmr', search_kwargs={'k':2, 'fetch_k':5}
)

@tool
def rag_tool(query:str):
    "Retrieves data about crow and fox story"
    retrieved_documents = retriever.invoke(query)
    content = "\\n".join([doc.page_content for doc in retrieved_documents])
    return content


@tool
def web_search_tool(query:str):
    "Performs web search for the given query"
    results = tavily_tool.invoke(query)
    return "\n\n".join([res['content'] for res in results['results']])

tools = [rag_tool, web_search_tool]

In [36]:
#-----------------------#
agent = create_agent(model=llm,
                     tools=tools,
                     system_prompt="""
                     You are a smart agent. You are capable of doing web search, RAG tool
                     Use:
                     RAG tool -> when query is related to fox and crow
                     Web Search tool -> when query is demanding current information from web results.
                     Don't use tool wherever not need
                     """)

queries = [
    'Whats the weather in chennai?',
    'Why did fox decieve the crow',
    'Who is Graham bell'
]

for query in queries:
    print(f"--------{query.upper()}---------")
    inputs = {'messages':HumanMessage(content=query)}
    for chunk in agent.stream(inputs, stream_mode='updates'):
        print(chunk)

--------WHATS THE WEATHER IN CHENNAI?---------
{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 137, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_02ae59e84c', 'id': 'chatcmpl-EFeiAaVCdf1DJ1hviyP9N4QLpPGWy', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a02959-69cb-7ba3-8224-51a4c33a6a1a-0', tool_calls=[{'name': 'web_search_tool', 'args': {'query': 'current weather in Chennai'}, 'id': 'call_SQCmcjoKZJoO0i0Sy58Bjxbt', 'type': 'tool_call'}], invalid_tool_calls=[], usage_meta

TypeError: string indices must be integers, not 'str'

In [1]:
import datetime

In [2]:
datetime.datetime(2025,12,5)

datetime.datetime(2025, 12, 5, 0, 0)

In [17]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import Optional
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
load_dotenv()

llm = ChatOpenAI(model='gpt-4o-mini')

class Sentiment(BaseModel):
    sentiment: Optional[str] = Field(
        description='One of: postive, negative, neutral'
    )
    confidence : Optional[float] = Field(
        ge=0,
        le=1,
        description="Confidence score between o and 1"
    )

class Extraction(BaseModel):
    
    name : Optional[str] = Field(
        description="Name of the user"
        )
    date : Optional[str] = Field(
        description="Date mentioned by the user. Strictly format date as YYYY-MM-DD"
        )
    amount : Optional[int|float] = Field(
        description='Amount mentioned by the user. return integer or float value'
        )
    sentiment_with_confidence : Optional[Sentiment] = Field(
        description='Extract sentiment and confidence from users text'
        )
    
llm_with_structure_output = llm.with_structured_output(Extraction)

def run(query:str):
    return llm_with_structure_output.invoke([HumanMessage(content=query)])


if __name__ == "__main__":
    query = """His name is Hitler. Born in 22-05-1897
    He is a cruel person. He killed 1 million jews"""
    print(run(query=query))

name='Hitler' date='1897-05-22' amount=1000000 sentiment_with_confidence=Sentiment(sentiment='negative', confidence=0.95)
